# ITER Divertor Thermal Simulation

Steady-state and transient 1D heat conduction model of a fusion divertor, with an analytical steady-state check and a numerical (finite-difference) solution.

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 2. Physical Parameters

In [ ]:
# Physical parameters

q = 2e7          # Plasma heat flux (W/m^2)
L = 0.01         # Divertor thickness (m)
k = 340          # Thermal conductivity (W/m/K)
T_cool = 373     # Cooling temperature (K)

# Material properties (tungsten)
rho = 19250      # Density (kg/m^3)
cp = 134         # Specific heat capacity (J/kg/K)

alpha = k / (rho * cp)   # Thermal diffusivity (m^2/s)

print("Thermal diffusivity:", alpha, "m^2/s")

## 3. Analytical Steady-State Solution

$T(x) = T_{cool} + \dfrac{q''}{k}(L - x)$

In [ ]:
x = np.linspace(0, L, 100)
T_analytical_ss = T_cool + (q / k) * (L - x)

T_surface = T_analytical_ss[0]
print("Surface temperature:", T_surface, "K")
print("Surface temperature:", T_surface - 273.15, "°C")

plt.plot(x * 1000, T_analytical_ss)
plt.xlabel("Depth into divertor (mm)")
plt.ylabel("Temperature (K)")
plt.title("Steady-State Divertor Temperature Profile")
plt.grid()
plt.show()

## 4. Parameter Study (Analytical)

In [ ]:
# Effect of plasma heat flux
heat_fluxes = np.array([5e6, 1e7, 1.5e7, 2e7, 2.5e7])
surface_temperatures = T_cool + (heat_fluxes * L) / k

plt.plot(heat_fluxes / 1e6, surface_temperatures, marker='o')
plt.xlabel("Plasma heat flux (MW/m²)")
plt.ylabel("Surface temperature (K)")
plt.title("Effect of Plasma Heat Flux on Divertor Surface Temperature")
plt.grid()
plt.show()

In [ ]:
# Effect of thermal conductivity
thermal_conductivities = np.array([50, 100, 150, 200, 250, 300])
surface_temperatures_k = T_cool + (q * L) / thermal_conductivities

plt.plot(thermal_conductivities, surface_temperatures_k, marker='o')
plt.xlabel("Thermal conductivity k (W/m/K)")
plt.ylabel("Surface temperature (K)")
plt.title("Effect of Thermal Conductivity on Surface Temperature")
plt.grid()
plt.show()

In [ ]:
# Effect of divertor thickness
thicknesses = np.array([2, 4, 6, 8, 10, 12]) / 1000
surface_temperatures_L = T_cool + (q * thicknesses) / k

plt.plot(thicknesses * 1000, surface_temperatures_L, marker='o')
plt.xlabel("Divertor thickness (mm)")
plt.ylabel("Surface temperature (K)")
plt.title("Effect of Divertor Thickness on Surface Temperature")
plt.grid()
plt.show()

## 5. Numerical Model: Discretization & Stability

In [ ]:
# Numerical (transient) parameters -- using conservative case values
q = 1e7
L = 0.01
k = 170
T_cool = 373
alpha = k / (rho * cp)

N = 50                 # number of grid points
dt = 0.0001            # time step (s)
total_time = 10        # total simulated time (s)

x = np.linspace(0, L, N)
dx = x[1] - x[0]

# Stability check: r = alpha*dt/dx^2 <= 0.5
r = alpha * dt / dx**2

print("dx:", dx, "m")
print("dt:", dt, "s")
print("Stability factor r:", r)

## 6. Transient Finite-Difference Simulation

In [ ]:
# Explicit FTCS scheme with plasma heat-flux BC at x=0 and fixed cooling BC at x=L
T = np.full(N, T_cool, dtype=float)

profiles, profile_times = [], []
surface_history, time_history = [], []

number_of_steps = int(total_time / dt)

for n in range(number_of_steps):
    T_new = T.copy()

    # Interior: 1D heat conduction
    for i in range(1, N - 1):
        T_new[i] = T[i] + alpha * dt * (
            (T[i + 1] - 2*T[i] + T[i - 1]) / dx**2
        )

    # Plasma heat-flux boundary (x = 0)
    T_new[0] = T_new[1] + (q * dx) / k

    # Fixed cooling boundary (x = L)
    T_new[-1] = T_cool

    T = T_new

    surface_history.append(T[0])
    time_history.append((n + 1) * dt)

    if n in [0, 99, 499, 999, number_of_steps - 1]:
        profiles.append(T.copy())
        profile_times.append((n + 1) * dt)

print("Initial surface temperature:", surface_history[0], "K")
print("Final surface temperature:", surface_history[-1], "K")
print("Analytical steady-state temperature:", T_cool + q * L / k, "K")

## 7. Transient Results

In [ ]:
for profile, t in zip(profiles, profile_times):
    plt.plot(x * 1000, profile, label=f"t = {t:.3f} s")

plt.xlabel("Depth into divertor (mm)")
plt.ylabel("Temperature (K)")
plt.title("Transient Divertor Temperature Evolution")
plt.legend()
plt.grid()
plt.show()

In [ ]:
plt.plot(time_history, surface_history)
plt.xlabel("Time (s)")
plt.ylabel("Surface temperature (K)")
plt.title("Divertor Surface Temperature vs Time")
plt.grid()
plt.show()

## 8. Validation: Numerical vs Analytical

In [ ]:
T_analytical = T_cool + (q / k) * (L - x)

plt.plot(x * 1000, T, label="Numerical")
plt.plot(x * 1000, T_analytical, "--", label="Analytical")
plt.xlabel("Depth into divertor (mm)")
plt.ylabel("Temperature (K)")
plt.title("Numerical vs Analytical Steady-State Solution")
plt.legend()
plt.grid()
plt.show()

## 9. Error Analysis

In [ ]:
absolute_error = np.abs(T - T_analytical)

maximum_error = np.max(absolute_error)
mean_error = np.mean(absolute_error)
relative_error = maximum_error / np.max(T_analytical) * 100

print("Maximum absolute error:", maximum_error, "K")
print("Mean absolute error:", mean_error, "K")
print("Maximum relative error:", relative_error, "%")

plt.plot(x * 1000, absolute_error)
plt.xlabel("Depth into divertor (mm)")
plt.ylabel("Absolute error (K)")
plt.title("Numerical Error Across the Divertor")
plt.grid()
plt.show()

## 10. Grid Convergence

In [ ]:
grid_sizes = [25, 50, 100, 200]
convergence_results = []

for N_test in grid_sizes:
    x_test = np.linspace(0, L, N_test)
    dx_test = x_test[1] - x_test[0]

    # Stable time step for this grid (r = 0.4)
    dt_test = 0.4 * dx_test**2 / alpha

    T_test = np.full(N_test, T_cool, dtype=float)
    steps = int(total_time / dt_test)

    for n in range(steps):
        T_new = T_test.copy()
        for i in range(1, N_test - 1):
            T_new[i] = T_test[i] + alpha * dt_test * (
                (T_test[i + 1] - 2*T_test[i] + T_test[i - 1]) / dx_test**2
            )
        T_new[0] = T_new[1] + (q * dx_test) / k
        T_new[-1] = T_cool
        T_test = T_new

    analytical_surface = T_cool + q * L / k
    error = abs(T_test[0] - analytical_surface)
    convergence_results.append((N_test, T_test[0], error))

    print("N =", N_test, "| dx =", dx_test, "| dt =", dt_test,
          "| Numerical =", T_test[0], "K | Error =", error, "K")